# Customer Support Goal-Oriented Multi Agent Flow (CrewAI)

### **Problem Statement**
Customer support teams often receive diverse queries—from login issues and billing errors to technical problems. Manually analyzing each message, searching a knowledge base, generating a helpful response, and deciding whether to escalate takes time and is inconsistent across agents.

### **Approach**
This workflow uses a **goal-oriented multi-agent system** to automate support resolution. The flow dynamically analyzes the user's query using four cooperating AI agents:

1. **Issue Identification Agent**  
   Classifies the query by intent, urgency, and sentiment.

2. **Knowledge Search Agent**  
   Searches relevant internal documentation or solutions.

3. **Solution Generation Agent**  
   Drafts a clear, user-friendly support response.

4. **Escalation Agent**  
   Evaluates if the AI response is sufficient or requires human review.

User input (`customer_query`) drives the reasoning across agents, making the workflow adaptable to real-world support scenarios.

✔ Faster support  
✔ Personalized responses  
✔ Smart escalation logic


For running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [3]:
# !pip install crewai==0.28.8 crewai_tools==0.1.6


In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

2026-10-31 21:06:43,133 - 8676535872 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 32
2026-10-31 21:06:43,134 - 8676535872 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 33
2026-10-31 21:06:43,134 - 8676535872 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 34


In [3]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


2026-10-31 21:08:26,209 - 8676535872 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 32
2026-10-31 21:08:26,219 - 8676535872 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 33
2026-10-31 21:08:26,219 - 8676535872 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 34


## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: issue_identifier
### Agent: knowledgebase_searcher
### Agent: solution_generator
### Agent: escalation_agent

In [5]:

issue_classifier = Agent(
    role="Issue Identification Specialist",
    goal="Correctly interpret and classify customer issues.",
    backstory="You are trained in identifying user intent and support category.",
    llm = llm,
    verbose=True
)

knowledge_agent = Agent(
    role="Knowledge Base Specialist",
    goal="Search the internal documentation to find matching solutions.",
    backstory="You are a search system that returns the best match from the support knowledge base.",
    llm=llm,
    verbose=True
)

solution_agent = Agent(
    role="Solution Generator",
    goal="Craft a final, user-friendly response using available info.",
    backstory="You write empathetic support replies with clarity and helpful tone.",
    llm=llm,
    verbose=True
)

escalation_agent = Agent(
    role="Escalation Coordinator",
    goal="Determine whether escalation to a human agent is required.",
    backstory="You evaluate uncertainty, urgency, or missing details to decide escalation.",
    llm=llm,
    verbose=True
)


In [ ]:
# ---------------- Tasks ----------------
identify_issue = Task(
    description="""Analyze the customer message: "{customer_query}".
Identify:
- Issue type (billing, technical, login, refund, general inquiry)
- Sentiment (angry, neutral, confused, urgent)
- Priority level (low, normal, high)
Output as JSON with keys: type, urgency, sentiment, summary.""",
    agent=issue_classifier,
    expected_output="A JSON object describing classified issue details."
)


In [11]:
## Define new Intent Identification Tasks here 

In [ ]:
classify_issue = Task(
    description="""Classify the primary intent of the customer message below.

<customer_message>
{customer_query}
</customer_message>

Choose exactly one issue_type:
- billing: charges, invoices, payment failures, subscriptions, or pricing
- technical: product malfunction, crash, sync failure, or unexpected behavior
- login: authentication, passwords, account lockout, or account access
- refund: an explicit request to return money or reverse a charge
- general_inquiry: informational or product-capability questions without a malfunction

Rules:
1. Classify the customer's primary requested outcome.
2. Choose the most specific applicable category.
3. Use refund when money is explicitly requested back, even if billing is involved.
4. Use login instead of technical for authentication or account-access problems.
5. Do not infer facts or causes that are not stated in the message.

Examples:
Message: I was charged twice this month. Please return the extra payment.
Output: {\"issue_type\": \"refund\", \"reason\": \"The customer explicitly requests that a duplicate charge be returned.\"}

Message: Why is my subscription price higher this month?
Output: {\"issue_type\": \"billing\", \"reason\": \"The customer is asking about subscription pricing and has not requested a refund.\"}

Message: The app says my password is incorrect, but I reset it this morning.
Output: {\"issue_type\": \"login\", \"reason\": \"The customer cannot authenticate and access the account.\"}

Message: The dashboard crashes after I sign in.
Output: {\"issue_type\": \"technical\", \"reason\": \"Authentication succeeds, but the product crashes afterward.\"}

Return valid JSON only with exactly these keys:
{\"issue_type\": \"<label>\", \"reason\": \"<one concise sentence>\"}""",
    agent=issue_classifier,
    expected_output="Valid JSON containing exactly issue_type and reason."
)

analyze_sentiment = Task(
    description="""Analyze the emotional tone explicitly expressed in this message.

<customer_message>
{customer_query}
</customer_message>

Choose exactly one sentiment:
- positive: appreciative, satisfied, or complimentary
- neutral: factual, polite, or emotionally unmarked
- confused: explicitly expresses uncertainty or lack of understanding
- frustrated: expresses inconvenience, repeated difficulty, or dissatisfaction
- angry: uses strong hostility, accusations, threats, or aggressive language

Rules:
1. Judge only the language in the message.
2. Do not infer emotion from the severity of the issue.
3. A serious or urgent issue can still have neutral sentiment.
4. If multiple emotions are explicit, choose the strongest one.
5. Urgent is not a sentiment.

Examples:
Message: I cannot log in. Please help.
Output: {\"sentiment\": \"neutral\", \"reason\": \"The customer states the problem politely without explicit emotional language.\"}

Message: I don't understand which password I am supposed to use.
Output: {\"sentiment\": \"confused\", \"reason\": \"The customer explicitly communicates uncertainty.\"}

Message: This has failed three times, and I am getting really frustrated.
Output: {\"sentiment\": \"frustrated\", \"reason\": \"The customer explicitly describes repeated failure and frustration.\"}

Return valid JSON only with exactly these keys:
{\"sentiment\": \"<label>\", \"reason\": \"<one concise sentence>\"}""",
    agent=issue_classifier,
    expected_output="Valid JSON containing exactly sentiment and reason."
)

classify_priority = Task(
    description="""Determine the operational support priority of this message.

<customer_message>
{customer_query}
</customer_message>

Choose exactly one priority:
- low: informational question or minor inconvenience with no blocked workflow
- normal: one customer is blocked or has a standard product problem, with no stated major impact, security risk, or severe time pressure
- high: explicit security risk, financial harm, widespread outage, business-critical failure, data loss, safety concern, or severe time-sensitive impact

Rules:
1. Use only impact, scope, security risk, and time sensitivity stated in the message.
2. Do not infer business impact merely because a feature is unavailable.
3. Angry language alone does not make an issue high priority.
4. Words such as urgent or ASAP are supporting signals, not sufficient by themselves.
5. If material impact is unclear, choose normal rather than assuming the worst.
6. Suspected unauthorized transactions or account takeover are high priority.

Examples:
Message: I cannot log in. Can you help?
Output: {\"priority\": \"normal\", \"signals\": [\"single user cannot access account\"], \"reason\": \"The customer is blocked, but no severe impact or security concern is stated.\"}

Message: Nobody in our company can log in, and our support operation is down.
Output: {\"priority\": \"high\", \"signals\": [\"multiple users affected\", \"business operation is down\"], \"reason\": \"The issue has organization-wide, business-critical impact.\"}

Message: This is urgent! Where can I change my profile picture?
Output: {\"priority\": \"low\", \"signals\": [\"informational request\", \"no blocked critical workflow\"], \"reason\": \"The word urgent does not establish meaningful operational impact.\"}

Message: I received a login code that I did not request.
Output: {\"priority\": \"high\", \"signals\": [\"possible unauthorized access\"], \"reason\": \"An unexpected authentication code indicates a potential account security threat.\"}

Return valid JSON only with exactly these keys:
{\"priority\": \"<low|normal|high>\", \"signals\": [\"<explicit signal>\"], \"reason\": \"<one concise sentence>\"}""",
    agent=issue_classifier,
    expected_output="Valid JSON containing exactly priority, signals, and reason."
)

In [ ]:

fetch_knowledge = Task(
    description="""Based on the classification above and the message "{customer_query}",
search the knowledge base and retrieve the closest matching solution.
If no solution exists, respond with: 'NO MATCH FOUND'.""",
    agent=knowledge_agent,
    expected_output="Best possible KB solution or 'NO MATCH FOUND'."
)

generate_solution_response = Task(
    description="""Using the classification and retrieved KB content for: "{customer_query}",
create a friendly step-by-step customer support response that:
- Uses plain language
- Offers clear next steps
- Includes apology if needed""",
    agent=solution_agent,
    expected_output="A polished support response formatted for users."
)

apply_escalation_logic = Task(
    description="""Evaluate whether the AI response is sufficient.
Consider:
- Confidence level
- Urgency from issue classification
- Whether solution was 'NO MATCH FOUND'

Output either:
- `resolved` if handled successfully
- `requires_escalation` with explanation and escalation notes""",
    agent=escalation_agent,
    expected_output="Resolution status + justification."
)

In [7]:

# ---------------- Crew Setup ----------------
cs_crew = Crew(
    agents=[issue_classifier, knowledge_agent, solution_agent, escalation_agent],
    tasks=[identify_issue, fetch_knowledge, generate_solution_response, apply_escalation_logic],
    verbose=True
)

In [8]:
# ---------------- Run ----------------
# customer_query = input("\nEnter customer message: ")
customer_query= """I’m unable to log into my account even though I’m sure the password is correct. Can someone help?"""

result = cs_crew.kickoff(inputs={"customer_query": customer_query})



 [DEBUG]: == Working Agent: Issue Identification Specialist
 [INFO]: == Starting Task: Analyze the customer message: "I’m unable to log into my account even though I’m sure the password is correct. Can someone help?".
Identify:
- Issue type (billing, technical, login, refund, general inquiry)
- Sentiment (angry, neutral, confused, urgent)
- Priority level (low, normal, high)
Output as JSON with keys: type, urgency, sentiment, summary.


> Entering new CrewAgentExecutor chain...
The customer is experiencing an issue with logging into their account, which indicates a technical problem related to account access. The customer expresses certainty about their password being correct, suggesting they may be confused about the login process or encountering an unexpected error. The tone of the message does not indicate anger but rather a neutral or confused sentiment. Given that the customer is seeking help, this could be considered an urgent matter, as they are unable to access their account.



In [10]:
print("\n--- FINAL SYSTEM RESPONSE ---\n")
print(result)


--- FINAL SYSTEM RESPONSE ---

resolved

The AI response provides a thorough set of troubleshooting steps for customers unable to log into their accounts, addressing various potential issues such as username verification, account lockout, password complexity, two-factor authentication, browser issues, system errors, account suspension, and security software interference. 

Additionally, I confirmed that there have been no recent changes to the login process or two-factor authentication that are not covered in the AI response. The common error messages that customers may encounter, such as "Incorrect Password," "Account Locked," "Two-Factor Authentication Failed," "Account Suspended," and "User Not Found," were also identified. Including these messages in the response could enhance clarity for the customer.

However, the existing response is already comprehensive and actionable, and it does not indicate a 'NO MATCH FOUND' scenario. Therefore, it is sufficient to address the customer's 

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
# This is urgent. My business is down because your system won’t sync orders. Fix this ASAP.
# I was charged twice for my subscription this month. Please refund the extra amount.
# The app keeps crashing every time I try to open the dashboard. This started after the latest update.
# Does your software support exporting data to Excel? I can’t find the option anywhere.
